# Image analysis for curtain obscuration

1. Set up data access and the image processor.
2. Process the photo sets and export the results.
3. Inspect prepared images.
4. Run standalone image checks.

Run sections 1–2 in order for the main pipeline. Section 3 uses the prepared cabinet images. Each check in section 4 has its own input paths and crop settings. Use the headings to navigate when comments or saved outputs refer to earlier cell numbers.

## 1. Set up data access and the image processor

### Mount Google Drive in Colab

The photo paths point to `/content/drive/MyDrive/Master`. Run this cell first in Colab.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Configure the photo sets, references, and crop

Load the libraries and set the photo directory, cabinet and lamp-and-cabinet references, region of interest, and filter settings.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║          CURTAIN OBSCURATION ANALYSIS — COMPLETE PIPELINE  v2              ║
# ║          Master's Project — D65 Controlled Lighting                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# NOTEBOOK STRUCTURE — run cells in order:
#   Cell 1 — Imports & Configuration
#   Cell 2 — ImageProcessor class
#   Cell 3 — Run pipeline
#   Cell 4 — Metrics, plots, export
#
# ══════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — IMPORTS & CONFIGURATION                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from skimage.metrics import structural_similarity as ssim
from skimage.feature  import graycomatrix, graycoprops
from skimage.exposure import match_histograms
from scipy import fftpack
from scipy.optimize import least_squares
import pandas as pd
import os
import glob

# RAW support
try:
    import rawpy
    RAW_SUPPORT = True
except ImportError:
    RAW_SUPPORT = False
    print("rawpy not installed — RAW files will be skipped.")
    print("Install with:  pip install rawpy")

# ── Edit these between sessions ───────────────────────────────────────────────
REFERENCE_CABINET      = '/content/drive/MyDrive/Master/Images 9th June/No fabric cabinet.JPG'
REFERENCE_LAMP_CABINET = '/content/drive/MyDrive/Master/Images 9th June/No fabric lamp&cabinet.JPG'
IMAGE_FOLDER         = '/content/drive/MyDrive/Master/Images 9th June'
OUTPUT_FOLDER        = os.path.join(IMAGE_FOLDER, 'results')

# ── Processing parameters ─────────────────────────────────────────────────────
EDGE_METHOD          = 'canny'
LOWPASS_RADIUS       = 300
SPIKE_THRESHOLD      = 3.0
NEIGHBOURHOOD_RADIUS = 15
PROTECT_CENTER       = 50
DARK_LEVEL           = 0      # 0 for JPEG/PNG; ~8 for RAW decoded to 8-bit

# ── ROI crop — set to the fabric/object region only ───────────────────────────
# x1=left, y1=top, x2=right, y2=bottom
FABRIC_CROP = (1273, 1119, 2200, 1766)  # ← replace with your actual pixel coordinates
# ── RAW extensions ────────────────────────────────────────────────────────────
RAW_EXTENSIONS = ['.cr2', '.cr3', '.nef', '.arw', '.dng',
                  '.orf', '.raf', '.rw2', '.pef', '.srw']

### Define the image processor

Load the `ImageProcessor` class used to read images, calculate metrics, create plots, and export the Excel reports.

In [ ]:



# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — ImageProcessor CLASS                                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.chart import BarChart, Reference
import openpyxl
class ImageProcessor:
    """
    Full image analysis pipeline for curtain obscuration characterisation.
    Compares curtain images against a no-curtain reference under D65 lighting.

    Key fix vs previous version
    ───────────────────────────
    Transmission now uses pixel-wise median ratio instead of ratio of means.
    Mean-based transmission was diluted by neutral background pixels giving
    near-identical values across all channels even for strongly coloured
    fabrics.  The pixel-wise median correctly captures per-channel attenuation.
    """

    RAW_EXTS = set()  # populated in __init__ from RAW_EXTENSIONS

    def __init__(self, image_paths, reference_path):
        self.image_paths    = image_paths
        self.reference_path = reference_path
        self.reference_idx  = self._resolve_reference(reference_path)
        self.images_color   = []
        self.images         = []
        self.fft_shifts     = []
        self.results        = {}
        ImageProcessor.RAW_EXTS = {e.lower() for e in RAW_EXTENSIONS}

    # ── Helpers ───────────────────────────────────────────────────────────────
    def crop_to_roi(self, crop):
        x1, y1, x2, y2 = crop
        self.images       = [img[y1:y2, x1:x2] for img in self.images]
        self.images_color = [img[y1:y2, x1:x2] for img in self.images_color]
        print(f"Cropped all images to ROI: {x2-x1}×{y2-y1}px")

    def _resolve_reference(self, path):
        for i, p in enumerate(self.image_paths):
            if p == path or os.path.basename(p) == path:
                return i
        raise ValueError(f"Reference image not found:\n  {path}")

    def _name(self, idx):
        return os.path.splitext(os.path.basename(self.image_paths[idx]))[0]

    def _is_ref(self, idx):
        return idx == self.reference_idx

    def _match_size(self, img, ref):
        if img.shape[:2] != ref.shape[:2]:
            return cv2.resize(img, (ref.shape[1], ref.shape[0]))
        return img

    @staticmethod
    def _is_raw(path):
        return os.path.splitext(path)[1].lower() in ImageProcessor.RAW_EXTS

    @staticmethod
    def _load_raw_as_bgr(path):
        if not RAW_SUPPORT:
            print(f"  rawpy not available — skipping: {os.path.basename(path)}")
            return None, None
        with rawpy.imread(path) as raw:
            rgb16 = raw.postprocess(use_camera_wb=True, output_bps=16, no_auto_bright=True)
        rgb8  = (rgb16 / 256).astype(np.uint8)
        bgr8  = cv2.cvtColor(rgb8, cv2.COLOR_RGB2BGR)
        gray8 = cv2.cvtColor(bgr8, cv2.COLOR_BGR2GRAY)
        return bgr8, gray8

    # ── Transmission (pixel-wise median — fixes mean dilution bug) ────────────

    @staticmethod

    def _beer_lambert_transmission(img_channel, ref_channel, dark_level=0.0):
        img_f = img_channel.astype(np.float32) - dark_level
        ref_f = ref_channel.astype(np.float32) - dark_level

        valid = ref_f > 10.0

        if valid.sum() < 100:
            r = max(float(ref_f.mean()), 1e-6)
            t = max(float(img_f.mean()), 0) / r
            if t > 1.0:
                print(f"  WARNING: fallback transmission = {t:.3f} > 1.0 "
                      f"(check lighting reference or alignment)")
            return float(t)

        ratio = np.clip(img_f[valid] / ref_f[valid], 0.0, None)  # no upper ceiling
        t = float(np.median(ratio))
        if t > 1.0:
            print(f"  WARNING: transmission = {t:.3f} > 1.0 "
                  f"(check lighting reference or alignment)")
        return t

    # ── Stage 1: Load ─────────────────────────────────────────────────────────

    def load_images(self):
        failed = 0
        for path in self.image_paths:
            if self._is_raw(path):
                color, gray = self._load_raw_as_bgr(path)
            else:
                color = cv2.imread(path, cv2.IMREAD_COLOR)
                gray  = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if color is not None:
                self.images_color.append(color)
                self.images.append(gray)
            else:
                print(f"  Could not load: {path}"); failed += 1
        print(f"Loaded {len(self.images)} images  ({failed} failed)")
        print(f"Reference: {self._name(self.reference_idx)}")

    def align_images_to_reference(self):
        """
        Align all images to the reference using feature matching.
        Corrects for any sub-pixel or pixel-level camera movement
        between shots even with a tripod.

        Method:
            1. Detect ORB keypoints in reference and each curtain image
            2. Match keypoints between the two
            3. Estimate a homography (perspective transform)
            4. Warp curtain image onto reference coordinate system

        This should be called immediately after load_images() and
        before any other processing.
        """
        ref_gray  = self.images[self.reference_idx]
        ref_color = self.images_color[self.reference_idx]

        orb     = cv2.ORB_create(nfeatures=5000)
        kp_ref, des_ref = orb.detectAndCompute(ref_gray, None)
        matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

        aligned_gray  = []
        aligned_color = []

        for i, (img_gray, img_color) in enumerate(zip(self.images, self.images_color)):
            if self._is_ref(i):
                aligned_gray.append(img_gray)
                aligned_color.append(img_color)
                print(f"  [{self._name(i)}] reference — no alignment needed")
                continue

            kp_img, des_img = orb.detectAndCompute(img_gray, None)

            if des_img is None or len(kp_img) < 10:
                print(f"  [{self._name(i)}] not enough keypoints — skipping alignment")
                aligned_gray.append(img_gray)
                aligned_color.append(img_color)
                continue

            # Match and sort by distance
            matches = sorted(matcher.match(des_ref, des_img), key=lambda x: x.distance)

            # Use top 20% of matches for robustness
            good_matches = matches[:max(10, len(matches)//5)]

            if len(good_matches) < 4:
                print(f"  [{self._name(i)}] too few good matches — skipping alignment")
                aligned_gray.append(img_gray)
                aligned_color.append(img_color)
                continue

            pts_ref = np.float32([kp_ref[m.queryIdx].pt for m in good_matches])
            pts_img = np.float32([kp_img[m.trainIdx].pt for m in good_matches])

            # Find homography — maps curtain image coords to reference coords
            H, mask = cv2.findHomography(pts_img, pts_ref, cv2.RANSAC, 5.0)

            if H is None:
                print(f"  [{self._name(i)}] homography failed — skipping alignment")
                aligned_gray.append(img_gray)
                aligned_color.append(img_color)
                continue

            h, w = ref_gray.shape
            img_gray_aligned  = cv2.warpPerspective(img_gray,  H, (w, h))
            img_color_aligned = cv2.warpPerspective(img_color, H, (w, h))

            # Report how much movement was corrected
            inliers = mask.ravel().sum()
            shift   = np.sqrt((H[0,2])**2 + (H[1,2])**2)
            print(f"  [{self._name(i)}] aligned — "
                  f"shift={shift:.2f}px  inliers={inliers}/{len(good_matches)}")

            aligned_gray.append(img_gray_aligned)
            aligned_color.append(img_color_aligned)

        self.images       = aligned_gray
        self.images_color = aligned_color
        print("Image alignment complete")

    # ── Stage 2: Edge detection ───────────────────────────────────────────────

    def edge_detection(self, method='canny'):
        edges = []
        for img in self.images:
            if method == 'canny':
                edge = cv2.Canny(img, 80, 150)
            elif method == 'canny_sensitive':
                edge = cv2.Canny(img, 20, 80)
            elif method == 'sobel':
                sx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
                sy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
                edge = np.uint8(np.clip(cv2.magnitude(sx, sy), 0, 255))
            elif method == 'laplacian':
                sh = cv2.filter2D(img, -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))
                edge = np.uint8(np.clip(np.abs(cv2.Laplacian(sh, cv2.CV_64F, ksize=3)), 0, 255))
            elif method == 'auto':
                sh    = cv2.filter2D(img, -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))
                canny = cv2.Canny(sh, 30, 100)
                sx    = cv2.Sobel(sh, cv2.CV_64F, 1, 0, ksize=3)
                sy    = cv2.Sobel(sh, cv2.CV_64F, 0, 1, ksize=3)
                sobel = np.uint8(np.clip(cv2.magnitude(sx, sy), 0, 255))
                edge  = cv2.max(canny, sobel)
            edges.append(edge)
        self.results['edges'] = edges
        print(f"Edge detection complete — method: {method}")
        return edges

    # ── Stage 3: Fourier transform ────────────────────────────────────────────

    def fourier_transform(self):
        fft_results = []
        self.fft_shifts = []
        for img in self.images:
            f_shift   = fftpack.fftshift(fftpack.fft2(img))
            magnitude = 20 * np.log(np.abs(f_shift) + 1)
            fft_results.append(magnitude)
            self.fft_shifts.append(f_shift)
        self.results['fft'] = fft_results
        print("Fourier transform complete")
        return fft_results

    # ── Stage 4: Low-pass filter ──────────────────────────────────────────────

    def apply_lowpass_filter(self, radius=300):
        lowpass_images, lowpass_fft = [], []
        for i, (img, f_shift) in enumerate(zip(self.images, self.fft_shifts)):
            rows, cols = f_shift.shape
            crow, ccol = rows // 2, cols // 2
            Y, X = np.ogrid[:rows, :cols]
            mask = (np.sqrt((X-ccol)**2 + (Y-crow)**2) <= radius).astype(np.float32)
            f_filtered = f_shift * mask
            img_clean  = np.clip(
                np.abs(fftpack.ifft2(fftpack.ifftshift(f_filtered))), 0, 255
            ).astype(np.uint8)
            mag_filt   = 20 * np.log(np.abs(f_filtered) + 1)
            lowpass_images.append(img_clean)
            lowpass_fft.append(mag_filt)

            fig, axes = plt.subplots(1, 4, figsize=(22, 5))
            fig.suptitle(f'Low-Pass Filter  |  {self._name(i)}', fontsize=11, y=1.01)
            for ax, im, title in zip(axes,
                    [img, self.results['fft'][i], mag_filt, img_clean],
                    ['Original', 'FFT (original)', f'FFT after low-pass (r={radius}px)', 'Filtered']):
                ax.imshow(im, cmap='gray'); ax.set_title(title, fontsize=9); ax.axis('off')
            axes[2].add_patch(plt.Circle((ccol, crow), radius,
                                          color='red', fill=False, lw=1.5, ls='--'))
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_FOLDER, f'{self._name(i)}_lowpass.png'),
                        dpi=150, bbox_inches='tight')
            plt.show()

        self.results['lowpass_images'] = lowpass_images
        self.results['lowpass_fft']    = lowpass_fft
        print(f"Low-pass filter complete — {len(lowpass_images)} images")
        return lowpass_images

    # ── Stage 5: Adaptive notch filter ───────────────────────────────────────

    def auto_notch_filter(self, spike_threshold=3.0, neighbourhood_radius=15,
                      protect_center=50):
        autonotch_images, autonotch_fft = [], []

        # Load reference magnitude once — used as baseline for spike detection
        ref_magnitude = self.results['fft'][self.reference_idx]

        for i, (img, f_shift) in enumerate(zip(self.images, self.fft_shifts)):
            if self._is_ref(i):
                print(f"  [{self._name(i)}] reference — skipped")
                autonotch_images.append(img.copy())
                autonotch_fft.append(self.results['fft'][i])
                continue

            rows, cols  = f_shift.shape
            crow, ccol  = rows // 2, cols // 2
            magnitude   = self.results['fft'][i]
            Y, X        = np.ogrid[:rows, :cols]
            center_mask = np.sqrt((X-ccol)**2 + (Y-crow)**2) < protect_center

            # Key change: work on the DIFFERENCE from the reference spectrum.
            # Only bins that are elevated ABOVE the reference are candidates —
            # this ensures we only remove fabric-specific periodicity, not
            # structure shared with the reference.
            diff        = magnitude - ref_magnitude
            diff        = np.clip(diff, 0.0, None)     # only positive excesses
            outer_diff  = diff[~center_mask]
            spike_map   = (outer_diff.std() > 0) and \
                          (diff > outer_diff.mean() + spike_threshold * outer_diff.std()) \
                          & (~center_mask)

            f_fixed = f_shift.copy()
            for sy, sx in np.argwhere(spike_map):
                y0, y1 = max(0, sy-neighbourhood_radius), min(rows, sy+neighbourhood_radius+1)
                x0, x1 = max(0, sx-neighbourhood_radius), min(cols, sx+neighbourhood_radius+1)
                nb     = f_fixed[y0:y1, x0:x1]
                NY, NX = np.ogrid[:nb.shape[0], :nb.shape[1]]
                d      = np.sqrt((NX-(sx-x0))**2 + (NY-(sy-y0))**2)
                ring   = (d > 1) & (d <= neighbourhood_radius)
                f_fixed[sy, sx] = np.mean(nb[ring]) if ring.sum() > 0 else 0

            img_clean   = np.clip(
                np.abs(fftpack.ifft2(fftpack.ifftshift(f_fixed))), 0, 255
            ).astype(np.uint8)
            mag_fixed   = 20 * np.log(np.abs(f_fixed) + 1)
            spike_count = int(spike_map.sum())
            autonotch_images.append(img_clean)
            autonotch_fft.append(mag_fixed)
            print(f"  [{self._name(i)}] {spike_count} spikes removed")

            overlay = np.stack([magnitude / magnitude.max()] * 3, axis=-1)
            overlay[spike_map, 0] = 1.0
            overlay[spike_map, 1] = 0.0
            overlay[spike_map, 2] = 0.0

            fig, axes = plt.subplots(1, 5, figsize=(26, 5))
            fig.suptitle(
                f'Adaptive Notch  |  {self._name(i)}  |  {spike_count} spikes',
                fontsize=11, y=1.01)
            for ax, im, t in zip(axes,
                    [img, magnitude, overlay, mag_fixed, img_clean],
                    ['Original','FFT (original)','Spikes (red)','FFT after notch','Cleaned']):
                ax.imshow(im, cmap=None if im.ndim==3 else 'gray')
                ax.set_title(t, fontsize=9); ax.axis('off')
            axes[2].add_patch(plt.Circle((ccol, crow), protect_center,
                                          color='cyan', fill=False, lw=1.5, ls='--'))
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_FOLDER, f'{self._name(i)}_autonotch.png'),
                        dpi=150, bbox_inches='tight')
            plt.show()

        self.results['autonotch_images'] = autonotch_images
        self.results['autonotch_fft']    = autonotch_fft
        print("Adaptive notch filter complete")
        return autonotch_images

    # ── Stage 6: Metrics ──────────────────────────────────────────────────────

    def compute_all_metrics(self, dark_level=0.0):
        ref_gray  = self.images[self.reference_idx]
        ref_color = self.images_color[self.reference_idx]
        ref_lab   = cv2.cvtColor(ref_color, cv2.COLOR_BGR2LAB).astype(np.float32)
        ref_edges = self.results['edges'][self.reference_idx]
        ref_fft   = self.results['fft'][self.reference_idx]
        ref_notch = self.results['autonotch_images'][self.reference_idx]
        ref_glcm  = graycomatrix(ref_gray, distances=[1], angles=[0], normed=True)
        h, w      = ref_fft.shape
        crow, ccol = h//2, w//2
        Y, X      = np.ogrid[:h, :w]
        hf_mask   = np.sqrt((X-ccol)**2 + (Y-crow)**2) > PROTECT_CENTER
        ref_energy = (ref_fft[hf_mask]**2).sum()

        metrics = []
        for i in range(len(self.images)):
            name = self._name(i)
            if self._is_ref(i):
                metrics.append({
                    'image': name, 'is_reference': True,
                    'ssim_before': 1.0, 'ssim_after': 1.0, 'delta_e': 0.0,
                    'transmission_r': 1.0, 'transmission_g': 1.0, 'transmission_b': 1.0,
                    'transmission_ratio_rg': 1.0, 'transmission_ratio_rb': 1.0,
                    'edge_recovery': 1.0, 'spectral_energy': 1.0,
                    'glcm_contrast': 1.0, 'glcm_homogeneity': 1.0,
                })
                continue

            img_gray  = self._match_size(self.images[i],                      ref_gray)
            img_color = self._match_size(self.images_color[i],                ref_color)
            img_notch = self._match_size(self.results['autonotch_images'][i], ref_notch)
            img_edges = self._match_size(self.results['edges'][i],            ref_edges)
            img_lab   = cv2.cvtColor(img_color, cv2.COLOR_BGR2LAB).astype(np.float32)

            # SSIM
            ssim_before, _ = ssim(ref_gray,  img_gray,  full=True)
            ssim_after,  _ = ssim(ref_notch, img_notch, full=True)

            # Delta-E CIE76
            diff    = ref_lab - img_lab
            delta_e = float(np.mean(np.sqrt(np.sum(diff**2, axis=2))))

            # Transmission — pixel-wise median (fixes mean dilution bug)
            img_f = img_color.astype(np.float32)
            ref_f = ref_color.astype(np.float32)
            t_b = self._beer_lambert_transmission(img_f[:,:,0], ref_f[:,:,0], dark_level)
            t_g = self._beer_lambert_transmission(img_f[:,:,1], ref_f[:,:,1], dark_level)
            t_r = self._beer_lambert_transmission(img_f[:,:,2], ref_f[:,:,2], dark_level)

            # Channel ratios — reveal colour cast independent of opacity
            ratio_rg = round(t_r / t_g, 4) if t_g > 0 else 0
            ratio_rb = round(t_r / t_b, 4) if t_b > 0 else 0

            # Edge recovery
            edge_recovery = img_edges.sum() / ref_edges.sum() if ref_edges.sum() > 0 else 0

            # Spectral energy
            img_fft_i = self.results['fft'][i]
            if img_fft_i.shape != ref_fft.shape:
                img_fft_i = cv2.resize(img_fft_i, (w, h))
            spectral_energy = float((img_fft_i[hf_mask]**2).sum() / ref_energy) \
                              if ref_energy > 0 else 0

            # GLCM
            img_glcm     = graycomatrix(img_gray, distances=[1], angles=[0], normed=True)
            ref_c        = float(graycoprops(ref_glcm, 'contrast')[0,0])
            img_c        = float(graycoprops(img_glcm, 'contrast')[0,0])
            ref_h        = float(graycoprops(ref_glcm, 'homogeneity')[0,0])
            img_h        = float(graycoprops(img_glcm, 'homogeneity')[0,0])
            glcm_contrast    = img_c / ref_c if ref_c > 0 else 0
            glcm_homogeneity = img_h / ref_h if ref_h > 0 else 0

            metrics.append({
                'image': name, 'is_reference': False,
                'ssim_before':           round(ssim_before,       4),
                'ssim_after':            round(ssim_after,         4),
                'delta_e':               round(delta_e,            4),
                'transmission_r':        round(t_r,                4),
                'transmission_g':        round(t_g,                4),
                'transmission_b':        round(t_b,                4),
                'transmission_ratio_rg': ratio_rg,
                'transmission_ratio_rb': ratio_rb,
                'edge_recovery':         round(edge_recovery,      4),
                'spectral_energy':       round(spectral_energy,    4),
                'glcm_contrast':         round(glcm_contrast,      4),
                'glcm_homogeneity':      round(glcm_homogeneity,   4),
            })

        self.results['metrics'] = metrics
        print("Metrics computed")
        return metrics

    # ── Stage 7: SSIM correction variants ────────────────────────────────────

    def compute_ssim_histogram_matched(self):
        ref = self.images[self.reference_idx]
        scores = []
        for i, img in enumerate(self.images):
            if self._is_ref(i): scores.append(1.0); continue
            img = self._match_size(img, ref)
            score, _ = ssim(ref, match_histograms(img, ref).astype(np.uint8), full=True)
            scores.append(score)
            print(f"  [{self._name(i)}] SSIM histogram match: {score:.4f}")
        self.results['ssim_histmatch'] = scores
        return scores

    def compute_ssim_clahe(self):
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        ref   = clahe.apply(self.images[self.reference_idx])
        scores = []
        for i, img in enumerate(self.images):
            if self._is_ref(i): scores.append(1.0); continue
            img = self._match_size(img, self.images[self.reference_idx])
            score, _ = ssim(ref, clahe.apply(img), full=True)
            scores.append(score)
            print(f"  [{self._name(i)}] SSIM CLAHE: {score:.4f}")
        self.results['ssim_clahe'] = scores
        return scores

    def compute_ssim_tophat(self, kernel_size=15):
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        ref    = self.images[self.reference_idx]
        scores = []
        for i, img in enumerate(self.images):
            if self._is_ref(i): scores.append(1.0); continue
            img = self._match_size(img, ref)
            img_clean = cv2.subtract(img, cv2.morphologyEx(img, cv2.MORPH_TOPHAT, kernel))
            score, _  = ssim(ref, img_clean, full=True)
            scores.append(score)
            print(f"  [{self._name(i)}] SSIM top-hat: {score:.4f}")
        self.results['ssim_tophat'] = scores
        return scores

    def compute_ssim_full_correction(self, kernel_size=15):
        clahe  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        ref    = self.images[self.reference_idx]
        ref_eq = clahe.apply(ref)
        scores = []
        for i, img in enumerate(self.images):
            if self._is_ref(i): scores.append(1.0); continue
            img       = self._match_size(img, ref)
            img_c     = cv2.subtract(img, cv2.morphologyEx(img, cv2.MORPH_TOPHAT, kernel))
            img_m     = match_histograms(img_c, ref).astype(np.uint8)
            img_final = clahe.apply(img_m)
            score, _  = ssim(ref_eq, img_final, full=True)
            scores.append(score)
            print(f"  [{self._name(i)}] SSIM full correction: {score:.4f}")
        self.results['ssim_full_correction'] = scores
        return scores

    # ── Stage 8: Visualisations ───────────────────────────────────────────────

    def plot_per_image_summary(self):
        ref_color = self.images_color[self.reference_idx]
        for i in range(len(self.images)):
            if self._is_ref(i): continue
            m = next(x for x in self.results['metrics'] if x['image'] == self._name(i))
            fig = plt.figure(figsize=(24, 8))
            fig.suptitle(
                f"{self._name(i)}  |  SSIM {m['ssim_before']:.4f}→{m['ssim_after']:.4f}  "
                f"|  ΔE {m['delta_e']:.2f}  |  Edge {m['edge_recovery']:.3f}",
                fontsize=12, fontweight='bold')
            gs = gridspec.GridSpec(2, 6, figure=fig, hspace=0.35, wspace=0.3)

            def _s(ax, im, t, cmap='gray'):
                ax.imshow(im, cmap=cmap); ax.set_title(t, fontsize=9); ax.axis('off')

            _s(fig.add_subplot(gs[0,0]), self.images[i],                              'Original (gray)')
            _s(fig.add_subplot(gs[0,1]), self.results['edges'][i],                    'Edge map')
            _s(fig.add_subplot(gs[0,2]), self.results['fft'][i],                      'FFT magnitude')
            _s(fig.add_subplot(gs[0,3]), self.results['autonotch_images'][i],         'After notch')
            _s(fig.add_subplot(gs[0,4]),
               cv2.cvtColor(self.images_color[i], cv2.COLOR_BGR2RGB), 'Colour', None)
            _s(fig.add_subplot(gs[0,5]),
               cv2.cvtColor(ref_color,            cv2.COLOR_BGR2RGB), 'Reference', None)

            ax_t = fig.add_subplot(gs[1,0:2])
            vals = [m['transmission_b'], m['transmission_g'], m['transmission_r']]
            bars = ax_t.bar(['Blue','Green','Red'], vals,
                            color=['steelblue','seagreen','tomato'], alpha=0.8)
            ax_t.axhline(1.0, color='black', lw=1, ls='--')
            ax_t.set_ylim(0, max(1.3, max(vals)+0.1))
            ax_t.set_title('Transmission (pixel-wise median)', fontsize=9)
            ax_t.set_ylabel('T (fraction of reference)')
            for bar, val in zip(bars, vals):
                ax_t.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                          f'{val:.3f}', ha='center', fontsize=8)

            ax_m = fig.add_subplot(gs[1,2:4]); ax_m.axis('off')
            ax_m.text(0.05, 0.95,
                f"SSIM before notch : {m['ssim_before']:.4f}\n"
                f"SSIM after notch  : {m['ssim_after']:.4f}\n"
                f"Delta-E (CIE76)   : {m['delta_e']:.4f}\n"
                f"Transmission R    : {m['transmission_r']:.4f}\n"
                f"Transmission G    : {m['transmission_g']:.4f}\n"
                f"Transmission B    : {m['transmission_b']:.4f}\n"
                f"Ratio R/G         : {m['transmission_ratio_rg']:.4f}\n"
                f"Ratio R/B         : {m['transmission_ratio_rb']:.4f}\n"
                f"Edge recovery     : {m['edge_recovery']:.4f}\n"
                f"Spectral energy   : {m['spectral_energy']:.4f}\n"
                f"GLCM contrast     : {m['glcm_contrast']:.4f}\n"
                f"GLCM homogeneity  : {m['glcm_homogeneity']:.4f}",
                transform=ax_m.transAxes, fontsize=9, va='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

            ax_h = fig.add_subplot(gs[1,4:6])
            ref_r = self._match_size(ref_color, self.images_color[i])
            for c, lbl, col in [(0,'Blue','steelblue'),(1,'Green','seagreen'),(2,'Red','tomato')]:
                ax_h.plot(cv2.calcHist([ref_r],                [c], None, [256], [0,256]).flatten(),
                          color=col, alpha=0.4, lw=1.5, ls='--')
                ax_h.plot(cv2.calcHist([self.images_color[i]], [c], None, [256], [0,256]).flatten(),
                          color=col, alpha=0.9, lw=1.5, label=lbl)
            ax_h.set_title('Histograms (dashed=ref)', fontsize=9)
            ax_h.set_xlabel('Pixel value'); ax_h.set_ylabel('Count')
            ax_h.legend(fontsize=7)

            plt.savefig(os.path.join(OUTPUT_FOLDER, f'{self._name(i)}_summary.png'),
                        dpi=150, bbox_inches='tight')
            plt.show()

    def plot_ssim_comparison(self):
        keys = {'ssim_before':'Original','ssim_after':'After notch',
                'ssim_histmatch':'Histogram match','ssim_clahe':'CLAHE',
                'ssim_tophat':'Top-hat','ssim_full_correction':'Full correction'}
        available   = {k:v for k,v in keys.items() if k in self.results}
        non_ref_idx = [i for i in range(len(self.images)) if not self._is_ref(i)]
        names = [self._name(i) for i in non_ref_idx]
        x     = np.arange(len(names))
        width = 0.8 / len(available)
        colors = plt.cm.tab10(np.linspace(0, 1, len(available)))
        fig, ax = plt.subplots(figsize=(max(12, len(names)*2), 6))
        fig.suptitle('SSIM — Correction Methods Comparison', fontsize=13, fontweight='bold')
        for idx, ((key, label), color) in enumerate(zip(available.items(), colors)):
            scores = [self.results[key][i] for i in non_ref_idx]
            offset = (idx - len(available)/2) * width + width/2
            bars   = ax.bar(x+offset, scores, width, label=label, color=color, alpha=0.85)
            for bar, val in zip(bars, scores):
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                        f'{val:.3f}', ha='center', fontsize=6.5, rotation=90)
        ax.set_xticks(x); ax.set_xticklabels(names, rotation=45, ha='right')
        ax.set_ylabel('SSIM Score'); ax.set_ylim(0, 1.15)
        ax.axhline(1.0, color='black', lw=0.8, ls='--', alpha=0.4)
        ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_FOLDER, 'ssim_methods_comparison.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()

    def plot_metrics_comparison(self):
        non_ref = [m for m in self.results['metrics'] if not m['is_reference']]
        if not non_ref: return
        names = [m['image'] for m in non_ref]
        plots = [
            ('ssim_before',           'SSIM (before notch)',        'steelblue'),
            ('ssim_after',            'SSIM (after notch)',         'royalblue'),
            ('delta_e',               'Delta-E CIE76 ↓',           'tomato'),
            ('transmission_r',        'Transmission R',             'tomato'),
            ('transmission_g',        'Transmission G',             'seagreen'),
            ('transmission_b',        'Transmission B',             'steelblue'),
            ('transmission_ratio_rg', 'Ratio R/G (>1 = red cast)', 'darkorange'),
            ('transmission_ratio_rb', 'Ratio R/B (>1 = red cast)', 'goldenrod'),
            ('edge_recovery',         'Edge recovery',              'mediumpurple'),
            ('spectral_energy',       'Spectral energy ratio',      'teal'),
            ('glcm_contrast',         'GLCM contrast ratio',        'coral'),
            ('glcm_homogeneity',      'GLCM homogeneity ratio',     'olivedrab'),
        ]
        n_cols = 2; n_rows = (len(plots)+1)//2
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*3.5),
                                  constrained_layout=True)
        fig.suptitle('Curtain Obscuration — Full Metrics', fontsize=14, fontweight='bold')
        for ax, (key, label, color) in zip(axes.flatten(), plots):
            vals = [m[key] for m in non_ref]
            bars = ax.bar(names, vals, color=color, alpha=0.8)
            ax.set_title(label, fontsize=10)
            ax.set_ylim(0, max(1.2, max(vals)*1.15) if vals else 1.2)
            ax.tick_params(axis='x', rotation=45, labelsize=7)
            ax.axhline(1.0, color='black', lw=0.8, ls='--', alpha=0.5)
            for bar, val in zip(bars, vals):
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                        f'{val:.3f}', ha='center', fontsize=7)
        for ax in axes.flatten()[len(plots):]: ax.set_visible(False)
        plt.savefig(os.path.join(OUTPUT_FOLDER, 'metrics_comparison.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()

    def plot_radar_chart(self):
        non_ref = [m for m in self.results['metrics'] if not m['is_reference']]
        if not non_ref: return
        max_de = max(m['delta_e'] for m in non_ref) or 1
        labels = ['SSIM\nafter notch','Edge\nrecovery','Spectral\nenergy',
                  'GLCM\nhomogeneity','Transmission\n(mean)','1−ΔE\n(norm)']
        n = len(labels)
        angles = [k/n*2*np.pi for k in range(n)] + [0]
        fig, ax = plt.subplots(figsize=(8,8), subplot_kw=dict(polar=True))
        fig.suptitle('Curtain Obscuration — Radar Chart', fontsize=13, fontweight='bold')
        colors = plt.cm.tab10(np.linspace(0, 1, len(non_ref)))
        for m, color in zip(non_ref, colors):
            t_mean = (m['transmission_r']+m['transmission_g']+m['transmission_b'])/3
            vals   = [m['ssim_after'], m['edge_recovery'], m['spectral_energy'],
                      m['glcm_homogeneity'], t_mean, 1-(m['delta_e']/max_de)]
            vals   = [min(max(v,0),1.3) for v in vals] + [min(max(vals[0],0),1.3)]
            ax.plot(angles, vals, color=color, lw=2, label=m['image'])
            ax.fill(angles, vals, color=color, alpha=0.08)
        ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, size=9)
        ax.set_ylim(0, 1.3)
        ax.legend(loc='upper right', bbox_to_anchor=(1.35,1.15), fontsize=8)
        plt.savefig(os.path.join(OUTPUT_FOLDER, 'radar_chart.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()

    # ── Stage 9: Excel export ─────────────────────────────────────────────────

    def export_full_excel_report(self, filename='curtain_obscuration_report.xlsx'):


        path = os.path.join(OUTPUT_FOLDER, filename)
        wb   = openpyxl.Workbook()

        def thin():
            s = Side(style='thin')
            return Border(left=s, right=s, top=s, bottom=s)

        def hdr(cell, bg='1F3864'):
            cell.fill      = PatternFill(start_color=bg, end_color=bg, fill_type='solid')
            cell.font      = Font(color='FFFFFF', bold=True, size=11)
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border    = thin()

        def dat(cell, row_idx):
            cell.alignment = Alignment(horizontal='center', vertical='center')
            cell.border    = thin()
            if row_idx % 2 == 0:
                cell.fill = PatternFill(start_color='DCE6F1', end_color='DCE6F1',
                                        fill_type='solid')

        def cw(ws, col, w):
            ws.column_dimensions[get_column_letter(col)].width = w

        def cs(ws, rng, reverse=False):
            lo, hi = ('63BE7B','F8696B') if reverse else ('F8696B','63BE7B')
            ws.conditional_formatting.add(rng, ColorScaleRule(
                start_type='min', start_color=lo,
                mid_type='percentile', mid_value=50, mid_color='FFEB84',
                end_type='max', end_color=hi))

        def title(ws, text, span):
            ws.merge_cells(span)
            c = ws[span.split(':')[0]]
            c.value = text
            c.font  = Font(bold=True, size=14, color='1F3864')
            c.alignment = Alignment(horizontal='center', vertical='center')
            ws.row_dimensions[1].height = 30

        non_ref     = [m for m in self.results['metrics'] if not m['is_reference']]
        non_ref_idx = [i for i in range(len(self.images)) if not self._is_ref(i)]
        names       = [m['image'] for m in non_ref]

        # Sheet 1 — Full Metrics
        ws1 = wb.active; ws1.title = 'Full Metrics Summary'
        ws1.freeze_panes = 'B3'
        title(ws1, 'Curtain Obscuration Analysis — Full Metrics Summary', 'A1:N1')
        hdrs = ['Image','SSIM\noriginal','SSIM\nafter notch','Delta-E\nCIE76 ↓',
                'Trans R','Trans G','Trans B','Ratio\nR/G','Ratio\nR/B',
                'Edge\nRecovery','Spectral\nEnergy','GLCM\nContrast',
                'GLCM\nHomog.','Interpretation']
        ws1.row_dimensions[2].height = 40
        for c, h in enumerate(hdrs, 1):
            hdr(ws1.cell(row=2, column=c, value=h)); cw(ws1, c, 16)
        cw(ws1,1,32); cw(ws1,14,42)

        def interp(m):
            t = []
            t.append('High' if m['ssim_after']<0.5 else
                     'Moderate' if m['ssim_after']<0.75 else 'Low')
            t[-1] += ' obscuration'
            if m['delta_e']>10: t.append('strong colour shift')
            elif m['delta_e']>3: t.append('noticeable colour shift')
            if m['transmission_ratio_rg']>1.3: t.append('red-dominant cast')
            elif m['transmission_ratio_rg']<0.7: t.append('green-dominant cast')
            if m['edge_recovery']<0.5: t.append('poor edge recovery')
            elif m['edge_recovery']<0.8: t.append('partial edge recovery')
            t_mean = (m['transmission_r']+m['transmission_g']+m['transmission_b'])/3
            if t_mean < 0.5: t.append('low transmission')
            return ', '.join(t)

        for ri, m in enumerate(non_ref, start=3):
            ws1.row_dimensions[ri].height = 22
            row = [m['image'], m['ssim_before'], m['ssim_after'], m['delta_e'],
                   m['transmission_r'], m['transmission_g'], m['transmission_b'],
                   m['transmission_ratio_rg'], m['transmission_ratio_rb'],
                   m['edge_recovery'], m['spectral_energy'],
                   m['glcm_contrast'], m['glcm_homogeneity'], interp(m)]
            for ci, val in enumerate(row, 1):
                cell = ws1.cell(row=ri, column=ci, value=val)
                dat(cell, ri)
                if isinstance(val, float): cell.number_format = '0.0000'

        last1 = 2+len(non_ref)
        for cl, rev in [('B',False),('C',False),('D',True),
                        ('E',False),('F',False),('G',False),
                        ('H',False),('I',False),('J',False),
                        ('K',False),('L',False),('M',False)]:
            cs(ws1, f'{cl}3:{cl}{last1}', reverse=rev)

        # Sheet 2 — SSIM comparison
        ws2 = wb.create_sheet('SSIM Comparison')
        ws2.freeze_panes = 'B3'
        title(ws2, 'SSIM — All Correction Methods', 'A1:G1')
        sh2 = ['Image','Original','After Notch','Histogram Match',
               'CLAHE','Top-Hat','Full Correction']
        sk2 = ['ssim_before','ssim_after','ssim_histmatch',
               'ssim_clahe','ssim_tophat','ssim_full_correction']
        ws2.row_dimensions[2].height = 35
        for c, h in enumerate(sh2, 1):
            hdr(ws2.cell(row=2, column=c, value=h)); cw(ws2, c, 20)
        cw(ws2,1,32)
        for ri, (name, idx) in enumerate(zip(names, non_ref_idx), start=3):
            ws2.row_dimensions[ri].height = 22
            dat(ws2.cell(row=ri, column=1, value=name), ri)
            for ci, key in enumerate(sk2, 2):
                vlist = self.results.get(key, [])
                val   = vlist[idx] if idx < len(vlist) else 'N/A'
                cell  = ws2.cell(row=ri, column=ci, value=val)
                dat(cell, ri)
                if isinstance(val, float): cell.number_format = '0.0000'
        last2 = 2+len(non_ref)
        for cl in ['B','C','D','E','F','G']:
            cs(ws2, f'{cl}3:{cl}{last2}')
        ch2 = BarChart(); ch2.type='col'
        ch2.title='SSIM Comparison'; ch2.style=10; ch2.width=24; ch2.height=14
        cats2 = Reference(ws2, min_col=1, min_row=3, max_row=last2)
        for c in range(2,8):
            ch2.add_data(Reference(ws2,min_col=c,min_row=2,max_row=last2),
                         titles_from_data=True)
        ch2.set_categories(cats2); ws2.add_chart(ch2, f'A{last2+3}')

        # Sheet 3 — Transmission
        ws3 = wb.create_sheet('Transmission')
        ws3.freeze_panes = 'B3'
        title(ws3, 'Beer-Lambert Transmission  (pixel-wise median)  T=1.0 → fully transparent', 'A1:F1')
        sh3 = ['Image','Trans R','Trans G','Trans B','Mean T','R/G ratio']
        ws3.row_dimensions[2].height = 35
        for c, (h, bg) in enumerate(zip(sh3,
                ['1F3864','C00000','375623','1F3864','7030A0','FF6600']), 1):
            cell = ws3.cell(row=2, column=c, value=h)
            hdr(cell, bg=bg); cw(ws3, c, 18)
        cw(ws3,1,32)
        for ri, m in enumerate(non_ref, start=3):
            t_mean = round((m['transmission_r']+m['transmission_g']+m['transmission_b'])/3,4)
            row = [m['image'],m['transmission_r'],m['transmission_g'],
                   m['transmission_b'],t_mean,m['transmission_ratio_rg']]
            ws3.row_dimensions[ri].height = 22
            for ci, val in enumerate(row, 1):
                cell = ws3.cell(row=ri, column=ci, value=val)
                dat(cell, ri)
                if isinstance(val, float): cell.number_format = '0.0000'
        last3 = 2+len(non_ref)
        from openpyxl.formatting.rule import ColorScaleRule as CSR
        for cl in ['B','C','D','E']:
            ws3.conditional_formatting.add(f'{cl}3:{cl}{last3}', CSR(
                start_type='min', start_color='F8696B',
                mid_type='num', mid_value=1.0, mid_color='FFFFFF',
                end_type='max', end_color='63BE7B'))
        ch3 = BarChart(); ch3.type='col'
        ch3.title='Channel Transmission per Curtain'; ch3.style=10
        ch3.width=22; ch3.height=14
        cats3 = Reference(ws3, min_col=1, min_row=3, max_row=last3)
        for c in [2,3,4]:
            ch3.add_data(Reference(ws3,min_col=c,min_row=2,max_row=last3),
                         titles_from_data=True)
        ch3.set_categories(cats3); ws3.add_chart(ch3, f'A{last3+3}')

        # Sheet 4 — Definitions
        ws4 = wb.create_sheet('Metric Definitions')
        title(ws4, 'Metric Definitions & Interpretation Guide', 'A1:D1')
        for c, h in enumerate(['Metric','Range','Good Value','Description'],1):
            hdr(ws4.cell(row=2,column=c,value=h))
        cw(ws4,1,28); cw(ws4,2,15); cw(ws4,3,15); cw(ws4,4,75)
        ws4.row_dimensions[2].height = 25
        guide = [
            ('SSIM (original)',        '0–1',  '≈1',
             'Structural similarity combining luminance, contrast and structure. Low = high total distortion from curtain.'),
            ('SSIM (after notch)',     '0–1',  '≈1',
             'SSIM after adaptive notch filter removes periodic FFT spikes. Improvement shows how much distortion was periodic noise.'),
            ('SSIM (histogram match)', '0–1',  '≈1',
             'Removes global brightness shift caused by curtain before SSIM. Large improvement = curtain mainly darkens image uniformly.'),
            ('SSIM (CLAHE)',           '0–1',  '≈1',
             'Corrects spatially uneven illumination. Improvement = curtain causes local brightness variation.'),
            ('SSIM (top-hat)',         '0–1',  '≈1',
             'Suppresses fabric weave texture. Improvement = visible weave pattern was a major contributor to distortion.'),
            ('SSIM (full correction)', '0–1',  '≈1',
             'All corrections combined. Best estimate of recoverable structural information.'),
            ('Delta-E CIE76',         '0–∞',  '<3',
             'Mean perceptual colour difference in LAB space. <1 imperceptible, 1–3 noticeable, >3 clearly visible colour cast.'),
            ('Transmission R/G/B',    '0–1',  '≈1',
             'Beer-Lambert transmission per channel computed as pixel-wise median ratio. '
             'Correctly handles neutral background dilution that makes mean-based ratios unreliable. '
             'A red curtain shows T_R > T_G, T_B.'),
            ('Ratio R/G',             '0–∞',  '≈1',
             'T_R / T_G. >1 = red dominant (red curtain). <1 = green dominant. '
             'Identifies colour bias independent of overall opacity.'),
            ('Ratio R/B',             '0–∞',  '≈1',
             'T_R / T_B. >1 = red dominant. Together with R/G gives the full colour cast fingerprint.'),
            ('Edge Recovery',         '0–1+', '≈1',
             'Ratio of detected edge pixels to reference. <1 = curtain blurs structural detail.'),
            ('Spectral Energy Ratio', '0–1+', '≈1',
             'High-frequency FFT energy ratio. <1 = fine detail lost. >1 = curtain adds noise/texture.'),
            ('GLCM Contrast',         '0–∞',  '≈1',
             'Texture contrast ratio. <1 = fabric smooths texture. >1 = fabric adds coarser texture.'),
            ('GLCM Homogeneity',      '0–1',  '≈1',
             'Texture uniformity ratio. >1 = more uniform/blurrier. <1 = more heterogeneous than reference.'),
        ]
        for ri, (metric, rng, good, desc) in enumerate(guide, start=3):
            ws4.row_dimensions[ri].height = 50
            for ci, val in enumerate([metric,rng,good,desc],1):
                cell = ws4.cell(row=ri, column=ci, value=val)
                cell.alignment = Alignment(horizontal='left' if ci==4 else 'center',
                                           vertical='center', wrap_text=True)
                cell.border = thin()
                if ri % 2 == 0:
                    cell.fill = PatternFill(start_color='DCE6F1', end_color='DCE6F1',
                                            fill_type='solid')

        wb.save(path)
        print(f"Excel report saved to:\n  {path}")
        return path

    # ── Local transmission map (spatially-varying, vs. single global scalar) ──

    def compute_local_transmission_map(self, window_size=15, valid_thresh=10.0):
        """
        Spatially-varying transmission map t(x), computed by comparing local
        window means of each curtain image against the reference, per BGR
        channel — instead of the single global scalar from
        _beer_lambert_transmission(). Reveals the fabric weave's local
        opacity variation (e.g. denser at seams/folds) that a single median
        value averages away.

        t(x) = boxFilter(I_curtain, window)(x) / boxFilter(I_ref, window)(x)

        Pixels where the local reference brightness is below `valid_thresh`
        are masked as NaN (same guard used in _beer_lambert_transmission) to
        avoid unstable ratios from near-zero denominators.
        """
        ref = self.images_color[self.reference_idx].astype(np.float32)
        ref_blur = cv2.boxFilter(ref, ddepth=-1, ksize=(window_size, window_size))

        maps = []
        for img in self.images_color:
            img_f = img.astype(np.float32)
            img_blur = cv2.boxFilter(img_f, ddepth=-1, ksize=(window_size, window_size))
            with np.errstate(divide='ignore', invalid='ignore'):
                t_map = np.where(ref_blur > valid_thresh,
                                  img_blur / np.maximum(ref_blur, 1e-6),
                                  np.nan)
            t_map = np.clip(t_map, 0.0, 1.5)  # small headroom above 1.0 for noise, not unbounded
            maps.append(t_map)

        self.results['local_transmission'] = maps
        print(f"Local transmission maps computed — window {window_size}×{window_size}")
        return maps

    def plot_local_transmission_maps(self, channel_names=('Blue', 'Green', 'Red')):
        """
        Visualise the local transmission map for each curtain image: one
        heatmap per BGR channel plus the mean across channels. Median values
        in each title should roughly match the global scalar from
        compute_all_metrics() — the point of this plot is the *spatial*
        pattern (uniform vs. patchy/weave-shaped opacity), not the number.
        """
        if 'local_transmission' not in self.results:
            raise RuntimeError("Run compute_local_transmission_map() first")

        for i in range(len(self.images_color)):
            if self._is_ref(i):
                continue
            t_map = self.results['local_transmission'][i]

            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            fig.suptitle(f"{self._name(i)} — Local Transmission Map",
                         fontsize=12, fontweight='bold')

            for c, (ax, name) in enumerate(zip(axes[:3], channel_names)):
                chan = np.ma.masked_invalid(t_map[..., c])
                im = ax.imshow(chan, cmap='inferno', vmin=0, vmax=1)
                ax.set_title(f'{name} channel  (median={np.nanmedian(t_map[..., c]):.3f})',
                             fontsize=9)
                ax.axis('off')
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

            mean_map = np.ma.masked_invalid(np.nanmean(t_map, axis=2))
            im = axes[3].imshow(mean_map, cmap='inferno', vmin=0, vmax=1)
            axes[3].set_title(f'Mean across channels  (median={np.nanmedian(mean_map):.3f})',
                               fontsize=9)
            axes[3].axis('off')
            fig.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_FOLDER, f'{self._name(i)}_local_transmission.png'),
                        dpi=150, bbox_inches='tight')
            plt.show()

    # ── Histogram comparison (dedicated, full-size — vs. the small subplot ────
    # ── panel already in plot_per_image_summary) ───────────────────────────────

    def plot_histogram_comparison(self, log_scale=False):
        """
        Full-size per-channel + grayscale intensity histogram of each curtain
        image against the no-curtain reference (dashed = reference). Larger
        and more detailed than the small histogram panel embedded in
        plot_per_image_summary — useful for reading off peak shifts, spread,
        and clipping at 0/255 precisely.
        """
        ref_color = self.images_color[self.reference_idx]
        ref_gray  = self.images[self.reference_idx]

        for i in range(len(self.images_color)):
            if self._is_ref(i):
                continue
            img_color = self.images_color[i]
            img_gray  = self.images[i]
            ref_c = self._match_size(ref_color, img_color)
            ref_g = self._match_size(ref_gray,  img_gray)

            fig, axes = plt.subplots(1, 4, figsize=(22, 5))
            fig.suptitle(f"{self._name(i)} — Histogram vs Reference (dashed = reference)",
                         fontsize=12, fontweight='bold')

            for ax, c, lbl, col in zip(axes[:3], [0, 1, 2], ['Blue', 'Green', 'Red'],
                                        ['steelblue', 'seagreen', 'tomato']):
                h_ref = cv2.calcHist([ref_c],     [c], None, [256], [0, 256]).flatten()
                h_img = cv2.calcHist([img_color], [c], None, [256], [0, 256]).flatten()
                ax.plot(h_ref, color=col, ls='--', alpha=0.6, lw=1.5, label='Reference')
                ax.plot(h_img, color=col, alpha=0.9, lw=1.5, label=self._name(i))
                if log_scale:
                    ax.set_yscale('log')
                ax.set_title(f'{lbl} channel', fontsize=9)
                ax.set_xlabel('Pixel value'); ax.set_ylabel('Count')
                ax.legend(fontsize=7)

            h_ref_g = cv2.calcHist([ref_g],    [0], None, [256], [0, 256]).flatten()
            h_img_g = cv2.calcHist([img_gray], [0], None, [256], [0, 256]).flatten()
            axes[3].plot(h_ref_g, color='black', ls='--', alpha=0.6, lw=1.5, label='Reference')
            axes[3].plot(h_img_g, color='black', alpha=0.9, lw=1.5, label=self._name(i))
            if log_scale:
                axes[3].set_yscale('log')
            axes[3].set_title('Grayscale', fontsize=9)
            axes[3].set_xlabel('Pixel value'); axes[3].set_ylabel('Count')
            axes[3].legend(fontsize=7)

            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_FOLDER, f'{self._name(i)}_histogram.png'),
                        dpi=150, bbox_inches='tight')
            plt.show()

    # ── Dark channel (He et al. Dark Channel Prior) ────────────────────────────

    def compute_dark_channel(self, window_size=15):
        """
        Dark Channel Prior: for each pixel, take the minimum intensity across
        B/G/R, then the minimum again over a local window (implemented as
        grayscale erosion, which is exactly a min-filter).

        In ordinary haze-free photos, this "dark channel" is close to 0
        almost everywhere — natural scenes almost always have *some* channel
        that's dark at *some* pixel nearby (shadows, saturated colours,
        dark surfaces). When a scattering medium sits in front of the scene
        (atmospheric haze, or here, curtain fabric), every channel gets
        pushed up toward the medium's own scattered colour, so the dark
        channel brightens. Its brightness is therefore a direct proxy for
        how much fabric-scattered light is dominating a region, independent
        of what colour the object underneath actually is.
        """
        maps = []
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (window_size, window_size))
        for img in self.images_color:
            min_channel = np.min(img.astype(np.float32), axis=2)
            dark = cv2.erode(min_channel, kernel)
            maps.append(dark)

        self.results['dark_channel'] = maps
        print(f"Dark channel computed — window {window_size}×{window_size}")
        return maps

    def estimate_airlight_from_dark_channel(self, top_percent=0.1):
        """
        Automatic estimate of A (the curtain's own scattered-light colour)
        per image, using the standard Dark Channel Prior heuristic: take the
        brightest `top_percent`% of pixels in the dark channel — the pixels
        most dominated by fabric-scattered light rather than the object's
        own colour — then average their *original* BGR values.

        This is the automatic counterpart to manually picking a "curtain
        only" corner patch: it finds the region of the curtain image the
        object contributes least to, without needing you to pick it by eye,
        and gives one A estimate per curtain (fabric), consistent with the
        haze-model recovery step: I(x) = J(x)·t(x) + A·(1 - t(x)).
        """
        if 'dark_channel' not in self.results:
            self.compute_dark_channel()

        A_estimates = []
        for img, dark in zip(self.images_color, self.results['dark_channel']):
            flat_dark = dark.flatten()
            n_pixels = max(1, int(len(flat_dark) * top_percent / 100))
            idx = np.argpartition(flat_dark, -n_pixels)[-n_pixels:]
            img_flat = img.reshape(-1, 3).astype(np.float32)
            A = img_flat[idx].mean(axis=0)  # B, G, R
            A_estimates.append(A)

        self.results['airlight_estimate'] = A_estimates
        return A_estimates

    def plot_dark_channel_comparison(self):
        """
        Dark channel of each curtain image next to the reference's dark
        channel, plus the mean dark-channel value for each (printed in the
        title). A higher mean than the reference indicates the fabric is
        adding uniformly-scattered light across the frame (a denser, more
        opaque weave); a low, patchy dark channel similar to the reference
        indicates a more open weave that still lets the object's true dark
        tones through in places.
        """
        if 'dark_channel' not in self.results:
            self.compute_dark_channel()

        ref_color = self.images_color[self.reference_idx]
        ref_dark  = self.results['dark_channel'][self.reference_idx]

        for i in range(len(self.images_color)):
            if self._is_ref(i):
                continue
            img_color = self.images_color[i]
            dark = self.results['dark_channel'][i]

            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            fig.suptitle(
                f"{self._name(i)} — Dark Channel Prior  "
                f"(mean dark value: reference={ref_dark.mean():.1f} → curtain={dark.mean():.1f})",
                fontsize=12, fontweight='bold')

            axes[0].imshow(cv2.cvtColor(ref_color, cv2.COLOR_BGR2RGB))
            axes[0].set_title('Reference', fontsize=9); axes[0].axis('off')

            axes[1].imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
            axes[1].set_title(self._name(i), fontsize=9); axes[1].axis('off')

            im2 = axes[2].imshow(ref_dark, cmap='inferno', vmin=0, vmax=255)
            axes[2].set_title('Reference dark channel', fontsize=9); axes[2].axis('off')
            fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

            im3 = axes[3].imshow(dark, cmap='inferno', vmin=0, vmax=255)
            axes[3].set_title('Curtain dark channel', fontsize=9); axes[3].axis('off')
            fig.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_FOLDER, f'{self._name(i)}_dark_channel.png'),
                        dpi=150, bbox_inches='tight')
            plt.show()




## 2. Process the photo sets and export results

### Load, crop, and filter both photo sets

Prepare the cabinet and lamp-and-cabinet images separately. Apply edge detection, a Fourier transform, and low-pass and notch filtering.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — RUN MAIN PIPELINE                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

std_exts = ['*.JPG','*.jpg','*.jpeg','*.JPEG','*.png','*.PNG']
raw_exts = [f'*{e}' for e in RAW_EXTENSIONS] + \
           [f'*{e.upper()}' for e in RAW_EXTENSIONS]

all_found = []
for ext in std_exts + (raw_exts if RAW_SUPPORT else []):
    all_found += glob.glob(os.path.join(IMAGE_FOLDER, ext))
all_found = sorted(set(all_found))

image_paths = all_found

if not image_paths:
    print("No images found — check IMAGE_FOLDER in Cell 1.")
elif REFERENCE_CABINET not in image_paths:
    print(f"Reference not found.\nExpected: {REFERENCE_CABINET}")
    print("Found:"); [print(f"  {p}") for p in image_paths]
else:
    print(f"Found {len(image_paths)} images")
    print(f"Output → {OUTPUT_FOLDER}\n")

    # --- Pass 1: cabinet-only images ---
    cabinet_images = [p for p in image_paths if 'lamp' not in os.path.basename(p).lower()]
    print(f"\n=== PASS 1: Cabinet ({len(cabinet_images)} images) ===")
    processor = ImageProcessor(cabinet_images, REFERENCE_CABINET)
    processor.load_images()
    processor.crop_to_roi(FABRIC_CROP)
    # processor.align_images_to_reference()   # commented out
    processor.edge_detection(method=EDGE_METHOD)
    processor.fourier_transform()
    processor.apply_lowpass_filter(radius=LOWPASS_RADIUS)
    processor.auto_notch_filter(
        spike_threshold=SPIKE_THRESHOLD,
        neighbourhood_radius=NEIGHBOURHOOD_RADIUS,
        protect_center=PROTECT_CENTER,
    )

    # --- Pass 2: lamp&cabinet images ---
    lamp_images = [p for p in image_paths if 'lamp' in os.path.basename(p).lower()]
    print(f"\n=== PASS 2: Lamp & Cabinet ({len(lamp_images)} images) ===")
    processor_lamp = ImageProcessor(lamp_images, REFERENCE_LAMP_CABINET)
    processor_lamp.load_images()
    processor_lamp.crop_to_roi(FABRIC_CROP)
    # processor_lamp.align_images_to_reference()
    processor_lamp.edge_detection(method=EDGE_METHOD)
    processor_lamp.fourier_transform()
    processor_lamp.apply_lowpass_filter(radius=LOWPASS_RADIUS)
    processor_lamp.auto_notch_filter(
        spike_threshold=SPIKE_THRESHOLD,
        neighbourhood_radius=NEIGHBOURHOOD_RADIUS,
        protect_center=PROTECT_CENTER,
    )

    print("\nBoth passes complete — run Cell 5 for metrics.")




Output hidden; open in https://colab.research.google.com to view.

### Calculate metrics and export the reports

Use both prepared photo sets to compare SSIM variants and other metrics. Save the plots and write `report_cabinet.xlsx` and `report_lamp_cabinet.xlsx`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — METRICS, VISUALISATIONS & EXPORT                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝


processor.compute_all_metrics(dark_level=DARK_LEVEL)
processor.compute_ssim_histogram_matched()
processor.compute_ssim_clahe()
processor.compute_ssim_tophat()
processor.compute_ssim_full_correction()
processor.plot_ssim_comparison()
processor.plot_per_image_summary()
processor.plot_metrics_comparison()
processor.plot_radar_chart()
excel_path = processor.export_full_excel_report(filename='report_cabinet.xlsx')

processor_lamp.compute_all_metrics(dark_level=DARK_LEVEL)
processor_lamp.compute_ssim_histogram_matched()
processor_lamp.compute_ssim_clahe()
processor_lamp.compute_ssim_tophat()
processor_lamp.compute_ssim_full_correction()
processor_lamp.plot_ssim_comparison()
processor_lamp.plot_per_image_summary()
processor_lamp.plot_metrics_comparison()
processor_lamp.plot_radar_chart()
excel_path_lamp = processor_lamp.export_full_excel_report(filename='report_lamp_cabinet.xlsx')

print("\nMain results saved to:", OUTPUT_FOLDER)




Output hidden; open in https://colab.research.google.com to view.

## 3. Inspect the prepared images

### Map local transmission and compare histograms and dark channels

Use the prepared cabinet images to plot per-channel transmission and intensity distributions, inspect dark channels, and estimate airlight.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — LOCAL TRANSMISSION MAP, HISTOGRAMS & DARK CHANNEL                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# Run after Cell 4 (needs processor.images / images_color already loaded,
# aligned, and cropped). Independent of the metrics/Excel export in Cell 4 —
# safe to re-run on its own.

# 1. Local transmission map — spatially-varying opacity, per channel
processor.compute_local_transmission_map(window_size=15)
processor.plot_local_transmission_maps()

# 2. Histogram of each curtain image vs. the reference
processor.plot_histogram_comparison()

# 3. Dark channel — what it says about each image:
#    A brighter dark channel than the reference means the curtain is adding
#    uniformly-scattered light across the frame (a denser, more opaque
#    weave). It's also how A (the curtain's own scattered-light colour) is
#    estimated automatically for haze-model recovery, using the brightest
#    dark-channel pixels instead of manually picking a "curtain only" patch.
processor.compute_dark_channel(window_size=15)
processor.plot_dark_channel_comparison()

A_estimates = processor.estimate_airlight_from_dark_channel(top_percent=0.1)
for i, A in enumerate(A_estimates):
    if processor._is_ref(i):
        continue
    print(f"{processor._name(i):30s}  A (B,G,R) ≈ {A[0]:.1f}, {A[1]:.1f}, {A[2]:.1f}")


## 4. Run standalone image checks

### Preview the crop

Display the selected region on the original images alongside the cropped views.

In [ ]:
"""
show_cropped_images.py

Standalone diagnostic — visualizes the FABRIC_CROP region against the
reference and fabric images, so you can confirm the crop is landing on
the right area before trusting mean-intensity or transmission numbers.

For each image, shows:
  - the full image with the crop rectangle drawn on it
  - the cropped result on its own

Usage:
    Edit the CONFIG section below, then run in Colab (plots will display inline)
    or locally with matplotlib installed.
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# ------------------------- CONFIG -------------------------

REFERENCE_PATH = "/content/drive/MyDrive/Master/Images 11th July fabric background/No fabric cabinet.JPG"

FABRIC_PATHS = [
    "/content/drive/MyDrive/Master/Images 11th July fabric background/No fabric lamp&cabinet.JPG",
    "/content/drive/MyDrive/Master/Images 11th July fabric background/White cabinet.JPG",
    "/content/drive/MyDrive/Master/Images 11th July fabric background/White lamp&cabinet.JPG",
]

FABRIC_CROP = (825, 914, 2700, 2200)  # (x1, y1, x2, y2)

# ------------------------------------------------------------


def show_image_with_crop(path, crop):
    if not os.path.exists(path):
        print(f"[not found] {path}")
        return

    img = Image.open(path).convert("RGB")
    arr = np.asarray(img)
    x1, y1, x2, y2 = crop
    cropped = arr[y1:y2, x1:x2, :]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(arr)
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor="red", facecolor="none"
    )
    axes[0].add_patch(rect)
    axes[0].set_title(f"Full image + crop box\n{os.path.basename(path)}")
    axes[0].axis("off")

    axes[1].imshow(cropped)
    axes[1].set_title(f"Cropped region\n{crop}")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


def main():
    print("Reference image:")
    show_image_with_crop(REFERENCE_PATH, FABRIC_CROP)

    for path in FABRIC_PATHS:
        show_image_with_crop(path, FABRIC_CROP)


if __name__ == "__main__":
    main()


Output hidden; open in https://colab.research.google.com to view.

### Check mean RGB intensity

Compare each selected fabric image’s mean red, green, and blue intensities with the no-fabric reference.

In [2]:
"""
mean_intensity_check.py

Standalone diagnostic — separate from the main pipeline notebook.

Computes the mean pixel intensity of the R, G, B channels for:
  - the reference (no-fabric) image
  - each fabric image

...and reports the ratio (fabric mean / reference mean) per channel.

This is meant to test whether transmission > 1 is due to the fabric
*redistributing* light (means stay close to reference) vs the fabric
image genuinely being brighter overall on average (mean ratio > 1).

Usage:
    Edit the CONFIG section below, then run:
        python mean_intensity_check.py
    or paste the whole file into its own Colab cell.
"""

import os
import numpy as np
from PIL import Image

# ------------------------- CONFIG -------------------------

# Path to the reference (no-fabric) image for this lighting condition
REFERENCE_PATH = "/content/drive/MyDrive/Master/Images 16th August/No fabric cabinet.JPG"   # or REFERENCE_LAMP_CABINET.jpg

# Paths to one or more fabric images to check against that reference
FABRIC_PATHS = [
    "/content/drive/MyDrive/Master/Images 16th August/2 Red 4 white cabinet.JPG",
    "/content/drive/MyDrive/Master/Images 16th August/Blue cabinet.JPG",
    "/content/drive/MyDrive/Master/Images 16th August/Brown cabinet.JPG",
    "/content/drive/MyDrive/Master/Images 16th August/Red cabinet.JPG",
    "/content/drive/MyDrive/Master/Images 16th August/White cabinet.JPG",
    "/content/drive/MyDrive/Master/Images 16th August/Yellow cabinet.JPG"
]
FABRIC_CROP = (1273, 1119, 2226, 1766)

# ------------------------------------------------------------


def load_and_crop(path, crop):
    img = Image.open(path).convert("RGB")
    arr = np.asarray(img).astype(np.float64)  # H x W x 3 (R, G, B)
    if crop is not None:
        x1, y1, x2, y2 = crop
        arr = arr[y1:y2, x1:x2, :]
    return arr


def mean_channels(arr):
    """Return (mean_R, mean_G, mean_B) over all pixels in the array."""
    return arr[..., 0].mean(), arr[..., 1].mean(), arr[..., 2].mean()


def main():
    if not os.path.exists(REFERENCE_PATH):
        raise FileNotFoundError(f"Reference image not found: {REFERENCE_PATH}")

    ref_arr = load_and_crop(REFERENCE_PATH, FABRIC_CROP)
    ref_r, ref_g, ref_b = mean_channels(ref_arr)

    print(f"Reference: {REFERENCE_PATH}")
    print(f"  Crop: {FABRIC_CROP if FABRIC_CROP else 'full image'}")
    print(f"  Mean R = {ref_r:8.3f}   Mean G = {ref_g:8.3f}   Mean B = {ref_b:8.3f}")
    print(f"  Mean overall (all channels) = {(ref_r + ref_g + ref_b) / 3:8.3f}")
    print()

    header = f"{'Fabric image':<30}{'Mean R':>10}{'Mean G':>10}{'Mean B':>10}   {'Ratio R':>8}{'Ratio G':>8}{'Ratio B':>8}"
    print(header)
    print("-" * len(header))

    for path in FABRIC_PATHS:
        if not os.path.exists(path):
            print(f"{path:<30}  [file not found]")
            continue

        arr = load_and_crop(path, FABRIC_CROP)
        r, g, b = mean_channels(arr)

        ratio_r = r / ref_r if ref_r else float("nan")
        ratio_g = g / ref_g if ref_g else float("nan")
        ratio_b = b / ref_b if ref_b else float("nan")

        name = os.path.basename(path)
        print(f"{name:<30}{r:10.3f}{g:10.3f}{b:10.3f}   {ratio_r:8.3f}{ratio_g:8.3f}{ratio_b:8.3f}")

    print()
    print("Interpretation:")
    print("  Ratio ~1.0        -> fabric mean brightness matches reference (redistribution, not net gain)")
    print("  Ratio consistently >1.0 across images -> fabric images are genuinely brighter on average")
    print("                                            (check exposure/ISO consistency, ambient light leaks,")
    print("                                             or whether the crop still includes bright regions)")


if __name__ == "__main__":
    main()

Reference: /content/drive/MyDrive/Master/Images 16th August/No fabric cabinet.JPG
  Crop: (1273, 1119, 2226, 1766)
  Mean R =   96.333   Mean G =   96.675   Mean B =   97.383
  Mean overall (all channels) =   96.797

Fabric image                      Mean R    Mean G    Mean B    Ratio R Ratio G Ratio B
---------------------------------------------------------------------------------------
2 Red 4 white cabinet.JPG        111.139   108.724   110.059      1.154   1.125   1.130
Blue cabinet.JPG                 126.341   125.969   125.590      1.312   1.303   1.290
Brown cabinet.JPG                113.980   112.979   111.862      1.183   1.169   1.149
Red cabinet.JPG                  124.636   124.079   123.409      1.294   1.283   1.267
White cabinet.JPG                124.558   124.125   122.821      1.293   1.284   1.261
Yellow cabinet.JPG               127.845   127.050   126.355      1.327   1.314   1.297

Interpretation:
  Ratio ~1.0        -> fabric mean brightness matches referenc

### Estimate median per-channel transmission

Calculate the median pixel-by-pixel fabric-to-reference ratio for each channel in the crop. Save `transmission_report.xlsx`.

In [ ]:
"""
median_transmission.py

Standalone diagnostic — replicates the main pipeline's transmission
calculation exactly, separate from the notebook:

  T = median over all pixels of (fabric_pixel / reference_pixel)

This matches the fixed pipeline logic from the session notes:
  - no hidden np.clip(..., 0.0, 2.0) ceiling
  - only clipped at 0.0 (np.clip(ratio, 0.0, None))
  - warns if T > 1.0 instead of silently capping it
  - computed on the cropped fabric ROI

Usage:
    Edit the CONFIG section below, then run in Colab or locally.
"""

import os
import numpy as np
from PIL import Image

# ------------------------- CONFIG -------------------------

REFERENCE_CABINET      = '/content/drive/MyDrive/Master/Images 11th July fabric background/No fabric cabinet.JPG'
REFERENCE_LAMP_CABINET = '/content/drive/MyDrive/Master/Images 11th July fabric background/No fabric lamp&cabinet.JPG'
IMAGE_FOLDER            = '/content/drive/MyDrive/Master/Images 11th July fabric background'

FABRIC_CROP = (825, 914, 2700, 2200)  # (x1, y1, x2, y2) -- corrected crop

MIN_REF_VALUE = 1.0  # avoid divide-by-zero on near-black reference pixels

EXCEL_OUTPUT_PATH = "/content/drive/MyDrive/Master/Images 11th July fabric background/transmission_report.xlsx"

# Filenames containing this (case-insensitive) are treated as lamp&cabinet
# condition images and compared against REFERENCE_LAMP_CABINET.
# Everything else is compared against REFERENCE_CABINET.
LAMP_CABINET_KEYWORD = "lamp"

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png")

# ------------------------------------------------------------


def discover_fabric_images(folder, reference_paths):
    """List every image in folder except the reference images themselves."""
    reference_basenames = {os.path.basename(p) for p in reference_paths}
    fabric_paths = []
    for name in sorted(os.listdir(folder)):
        if name in reference_basenames:
            continue
        if not name.lower().endswith(VALID_EXTENSIONS):
            continue
        fabric_paths.append(os.path.join(folder, name))
    return fabric_paths


def pick_reference(path):
    """Choose which reference image matches this fabric image's lighting condition."""
    name = os.path.basename(path).lower()
    if LAMP_CABINET_KEYWORD in name:
        return REFERENCE_LAMP_CABINET, "lamp&cabinet"
    return REFERENCE_CABINET, "cabinet"


def load_and_crop(path, crop):
    img = Image.open(path).convert("RGB")
    arr = np.asarray(img).astype(np.float64)
    if crop is not None:
        x1, y1, x2, y2 = crop
        arr = arr[y1:y2, x1:x2, :]
    return arr


def beer_lambert_transmission(fabric_arr, ref_arr, min_ref_value):
    """
    Per-pixel Beer-Lambert transmission, per channel.
    ratio = fabric / reference, clipped only at 0.0 (no upper ceiling).
    Pixels where reference < min_ref_value are excluded (NaN).
    """
    ratio = np.full_like(fabric_arr, np.nan)
    valid = ref_arr >= min_ref_value
    ratio[valid] = fabric_arr[valid] / ref_arr[valid]
    ratio = np.where(np.isnan(ratio), np.nan, np.clip(ratio, 0.0, None))
    return ratio


def median_transmission_per_channel(ratio, channel_names=("R", "G", "B")):
    result = {}
    for i, name in enumerate(channel_names):
        ch = ratio[..., i]
        valid = ch[~np.isnan(ch)]
        result[name] = np.median(valid)
    return result


def write_excel_report(rows, output_path):
    """
    rows: list of dicts with keys:
        fabric, condition, reference, mean_ratio,
        T_raw_R, T_raw_G, T_raw_B, T_norm_R, T_norm_G, T_norm_B
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, Alignment, PatternFill

    wb = Workbook()
    ws = wb.active
    ws.title = "Transmission Results"

    headers = [
        "Fabric", "Condition", "Reference image", "Mean ratio\n(fabric/reference)",
        "T raw R", "T raw G", "T raw B",
        "T normalized R", "T normalized G", "T normalized B",
    ]
    ws.append(headers)

    header_font = Font(name="Arial", bold=True)
    for cell in ws[1]:
        cell.font = header_font
        cell.alignment = Alignment(wrap_text=True, vertical="center")

    flag_fill = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")

    for row in rows:
        ws.append([
            row["fabric"], row["condition"], row["reference"], round(row["mean_ratio"], 3),
            round(row["T_raw_R"], 3), round(row["T_raw_G"], 3), round(row["T_raw_B"], 3),
            round(row["T_norm_R"], 3), round(row["T_norm_G"], 3), round(row["T_norm_B"], 3),
        ])
        r = ws.max_row
        for col_letter, val in zip(("E", "F", "G"), (row["T_raw_R"], row["T_raw_G"], row["T_raw_B"])):
            if val > 1.0:
                ws[f"{col_letter}{r}"].fill = flag_fill

    for row_cells in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row_cells:
            cell.font = Font(name="Arial")

    widths = [30, 14, 26, 16, 10, 10, 10, 16, 16, 16]
    for i, w in enumerate(widths, start=1):
        ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = w
    ws.row_dimensions[1].height = 30

    os.makedirs(os.path.dirname(output_path), exist_ok=True) if os.path.dirname(output_path) else None
    wb.save(output_path)
    print(f"Excel report saved to: {output_path}")
    print("Yellow-highlighted cells indicate raw T > 1.0")


def main():
    ref_arrays = {}
    ref_means = {}
    for label, path in (("cabinet", REFERENCE_CABINET), ("lamp&cabinet", REFERENCE_LAMP_CABINET)):
        arr = load_and_crop(path, FABRIC_CROP)
        ref_arrays[label] = arr
        ref_means[label] = arr.mean()
        print(f"Reference [{label}]: {os.path.basename(path)}")
        print(f"  Crop: {FABRIC_CROP}")
        print(f"  Overall mean intensity: {ref_means[label]:.3f}")
    print()

    fabric_paths = discover_fabric_images(
        IMAGE_FOLDER, [REFERENCE_CABINET, REFERENCE_LAMP_CABINET]
    )

    excel_rows = []

    for path in fabric_paths:
        ref_path, condition = pick_reference(path)
        ref_arr = ref_arrays[condition]
        ref_mean = ref_means[condition]

        fabric_arr = load_and_crop(path, FABRIC_CROP)
        fabric_mean = fabric_arr.mean()
        mean_ratio = fabric_mean / ref_mean  # single combined-channel scalar

        ratio = beer_lambert_transmission(fabric_arr, ref_arr, MIN_REF_VALUE)
        T_raw = median_transmission_per_channel(ratio)
        T_norm = {ch: val / mean_ratio for ch, val in T_raw.items()}

        print(f"Fabric: {os.path.basename(path)}  [condition: {condition}, ref: {os.path.basename(ref_path)}]")
        print(f"  Mean ratio (fabric_mean / reference_mean): {mean_ratio:.3f}")
        for ch in T_raw:
            raw_val = T_raw[ch]
            norm_val = T_norm[ch]
            flag = "  <-- T > 1.0" if raw_val > 1.0 else ""
            print(f"  T[{ch}] raw (median)        = {raw_val:.3f}{flag}")
            print(f"  T[{ch}] normalized (median) = {norm_val:.3f}")
        print()

        excel_rows.append({
            "fabric": os.path.basename(path),
            "condition": condition,
            "reference": os.path.basename(ref_path),
            "mean_ratio": mean_ratio,
            "T_raw_R": T_raw["R"], "T_raw_G": T_raw["G"], "T_raw_B": T_raw["B"],
            "T_norm_R": T_norm["R"], "T_norm_G": T_norm["G"], "T_norm_B": T_norm["B"],
        })

    write_excel_report(excel_rows, EXCEL_OUTPUT_PATH)


if __name__ == "__main__":
    main()

Reference [cabinet]: No fabric cabinet.JPG
  Crop: (825, 914, 2700, 2200)
  Overall mean intensity: 89.735
Reference [lamp&cabinet]: No fabric lamp&cabinet.JPG
  Crop: (825, 914, 2700, 2200)
  Overall mean intensity: 92.744

Fabric: White cabinet.JPG  [condition: cabinet, ref: No fabric cabinet.JPG]
  Mean ratio (fabric_mean / reference_mean): 1.191
  T[R] raw (median)        = 1.180  <-- T > 1.0
  T[R] normalized (median) = 0.991
  T[G] raw (median)        = 1.178  <-- T > 1.0
  T[G] normalized (median) = 0.989
  T[B] raw (median)        = 1.168  <-- T > 1.0
  T[B] normalized (median) = 0.981

Fabric: White lamp&cabinet.JPG  [condition: lamp&cabinet, ref: No fabric lamp&cabinet.JPG]
  Mean ratio (fabric_mean / reference_mean): 1.165
  T[R] raw (median)        = 1.154  <-- T > 1.0
  T[R] normalized (median) = 0.991
  T[G] raw (median)        = 1.164  <-- T > 1.0
  T[G] normalized (median) = 0.999
  T[B] raw (median)        = 1.149  <-- T > 1.0
  T[B] normalized (median) = 0.986

Excel 